In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
PROCESSED_DIR = Path("../data/processed")
CHECKPOINT_DIR = Path("../outputs/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 4
EPOCHS = 20
LR = 1e-4
NUM_WORKERS = 0
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
class LandsatPatchDataset(Dataset):
    def __init__(self, processed_dir, split):
        self.input_dir = Path(processed_dir) / split / "inputs"
        self.target_dir = Path(processed_dir) / split / "targets"

        self.input_files = sorted(self.input_dir.glob("*.npy"))
        self.target_files = sorted(self.target_dir.glob("*.npy"))

        assert len(self.input_files) == len(self.target_files), "Input/target count mismatch"

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        inp = np.load(self.input_files[idx]).astype(np.float32)
        tgt = np.load(self.target_files[idx]).astype(np.float32)

        inp = torch.from_numpy(inp).permute(2, 0, 1)
        tgt = torch.from_numpy(tgt).permute(2, 0, 1)

        return inp, tgt

In [ ]:
train_dataset = LandsatPatchDataset(PROCESSED_DIR, "train")
val_dataset = LandsatPatchDataset(PROCESSED_DIR, "val")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

x, y = next(iter(train_loader))
print("Input batch:", x.shape)
print("Target batch:", y.shape)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class CrossAttentionFusion(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()

        hidden = max(channels // reduction, 8)

        self.query = nn.Conv2d(channels, hidden, 1)
        self.key = nn.Conv2d(channels, hidden, 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.attn_expand = nn.Conv2d(hidden, channels, 1)

        self.gamma = nn.Parameter(torch.zeros(1))
        self.out_conv = DoubleConv(channels * 2, channels)

    def forward(self, thermal_feat, physics_feat):
        q = self.query(thermal_feat)
        k = self.key(physics_feat)

        attention = torch.sigmoid(q * k)
        attention = self.attn_expand(attention)

        v = self.value(physics_feat)
        physics_attended = v * attention

        fused = thermal_feat + self.gamma * physics_attended
        fused = torch.cat([fused, physics_feat], dim=1)

        return self.out_conv(fused)


class CrossAttentionSpectraFusionNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.t1 = DoubleConv(1, 32)
        self.t2 = DoubleConv(32, 64)
        self.t3 = DoubleConv(64, 128)
        self.t4 = DoubleConv(128, 256)

        self.p1 = DoubleConv(4, 32)
        self.p2 = DoubleConv(32, 64)
        self.p3 = DoubleConv(64, 128)
        self.p4 = DoubleConv(128, 256)

        self.f1 = CrossAttentionFusion(32)
        self.f2 = CrossAttentionFusion(64)
        self.f3 = CrossAttentionFusion(128)
        self.f4 = CrossAttentionFusion(256)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, 3, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        thermal = x[:, 0:1]
        physics = x[:, 1:5]

        t1 = self.t1(thermal)
        p1 = self.p1(physics)
        f1 = self.f1(t1, p1)

        t2 = self.t2(self.pool(t1))
        p2 = self.p2(self.pool(p1))
        f2 = self.f2(t2, p2)

        t3 = self.t3(self.pool(t2))
        p3 = self.p3(self.pool(p2))
        f3 = self.f3(t3, p3)

        t4 = self.t4(self.pool(t3))
        p4 = self.p4(self.pool(p3))
        f4 = self.f4(t4, p4)

        bn = self.bottleneck(self.pool(f4))

        u4 = self.up4(bn)
        u4 = self.dec4(torch.cat([u4, f4], dim=1))

        u3 = self.up3(u4)
        u3 = self.dec3(torch.cat([u3, f3], dim=1))

        u2 = self.up2(u3)
        u2 = self.dec2(torch.cat([u2, f2], dim=1))

        u1 = self.up1(u2)
        u1 = self.dec1(torch.cat([u1, f1], dim=1))

        return self.activation(self.out(u1))

In [ ]:
model = CrossAttentionSpectraFusionNet().to(device)

summary(
    model,
    input_size=(BATCH_SIZE, 5, 128, 128),
    col_names=["input_size", "output_size", "num_params"],
)

In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.window_size = window_size

    def forward(self, pred, target):
        c1 = 0.01 ** 2
        c2 = 0.03 ** 2

        mu_x = F.avg_pool2d(pred, self.window_size, stride=1, padding=self.window_size // 2)
        mu_y = F.avg_pool2d(target, self.window_size, stride=1, padding=self.window_size // 2)

        sigma_x = F.avg_pool2d(pred * pred, self.window_size, stride=1, padding=self.window_size // 2) - mu_x ** 2
        sigma_y = F.avg_pool2d(target * target, self.window_size, stride=1, padding=self.window_size // 2) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred * target, self.window_size, stride=1, padding=self.window_size // 2) - mu_x * mu_y

        ssim = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (
            (mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2)
        )

        return 1.0 - ssim.mean()


class PhysicsConsistencyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, inputs, pred):
        ndvi = inputs[:, 1:2]
        ndwi = inputs[:, 2:3]
        ndbi = inputs[:, 3:4]

        red = pred[:, 0:1]
        green = pred[:, 1:2]
        blue = pred[:, 2:3]

        vegetation_mask = (ndvi > 0.60).float()
        water_mask = (ndwi > 0.45).float()
        urban_mask = (ndbi > 0.55).float()

        vegetation_loss = torch.mean(vegetation_mask * F.relu(red - green))
        water_loss = torch.mean(water_mask * F.relu(red - blue))
        urban_loss = torch.mean(urban_mask * torch.abs(red - green))

        return vegetation_loss + water_loss + urban_loss


class HybridLoss(nn.Module):
    def __init__(self, l1_weight=0.5, ssim_weight=0.4, physics_weight=0.1):
        super().__init__()

        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.physics = PhysicsConsistencyLoss()

        self.l1_weight = l1_weight
        self.ssim_weight = ssim_weight
        self.physics_weight = physics_weight

    def forward(self, inputs, pred, target):
        l1_loss = self.l1(pred, target)
        ssim_loss = self.ssim(pred, target)
        physics_loss = self.physics(inputs, pred)

        total_loss = (
            self.l1_weight * l1_loss
            + self.ssim_weight * ssim_loss
            + self.physics_weight * physics_loss
        )

        return total_loss, l1_loss, ssim_loss, physics_loss

In [ ]:
criterion = HybridLoss(
    l1_weight=0.5,
    ssim_weight=0.4,
    physics_weight=0.1
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

print("Using Cross-Attention + Hybrid Loss")
print("Loss = 0.5 L1 + 0.4 SSIM + 0.1 Physics")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss_sum = 0.0
    l1_loss_sum = 0.0
    ssim_loss_sum = 0.0
    physics_loss_sum = 0.0

    for inputs, targets in tqdm(loader, leave=False):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(inputs)

        total_loss, l1_loss, ssim_loss, physics_loss = criterion(
            inputs,
            outputs,
            targets
        )

        total_loss.backward()
        optimizer.step()

        total_loss_sum += total_loss.item()
        l1_loss_sum += l1_loss.item()
        ssim_loss_sum += ssim_loss.item()
        physics_loss_sum += physics_loss.item()

    n = len(loader)

    return {
        "total": total_loss_sum / n,
        "l1": l1_loss_sum / n,
        "ssim": ssim_loss_sum / n,
        "physics": physics_loss_sum / n,
    }


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    total_loss_sum = 0.0
    l1_loss_sum = 0.0
    ssim_loss_sum = 0.0
    physics_loss_sum = 0.0

    for inputs, targets in tqdm(loader, leave=False):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        outputs = model(inputs)

        total_loss, l1_loss, ssim_loss, physics_loss = criterion(
            inputs,
            outputs,
            targets
        )

        total_loss_sum += total_loss.item()
        l1_loss_sum += l1_loss.item()
        ssim_loss_sum += ssim_loss.item()
        physics_loss_sum += physics_loss.item()

    n = len(loader)

    return {
        "total": total_loss_sum / n,
        "l1": l1_loss_sum / n,
        "ssim": ssim_loss_sum / n,
        "physics": physics_loss_sum / n,
    }

In [ ]:
best_val_loss = float("inf")

history = {
    "train_total": [],
    "val_total": [],
    "train_l1": [],
    "val_l1": [],
    "train_ssim": [],
    "val_ssim": [],
    "train_physics": [],
    "val_physics": [],
}

start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")

    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_metrics = validate(
        model,
        val_loader,
        criterion,
        device
    )

    history["train_total"].append(train_metrics["total"])
    history["val_total"].append(val_metrics["total"])
    history["train_l1"].append(train_metrics["l1"])
    history["val_l1"].append(val_metrics["l1"])
    history["train_ssim"].append(train_metrics["ssim"])
    history["val_ssim"].append(val_metrics["ssim"])
    history["train_physics"].append(train_metrics["physics"])
    history["val_physics"].append(val_metrics["physics"])

    print(
        f"Train Total: {train_metrics['total']:.6f} | "
        f"L1: {train_metrics['l1']:.6f} | "
        f"SSIM: {train_metrics['ssim']:.6f} | "
        f"Physics: {train_metrics['physics']:.6f}"
    )

    print(
        f"Val Total:   {val_metrics['total']:.6f} | "
        f"L1: {val_metrics['l1']:.6f} | "
        f"SSIM: {val_metrics['ssim']:.6f} | "
        f"Physics: {val_metrics['physics']:.6f}"
    )

    if val_metrics["total"] < best_val_loss:
        best_val_loss = val_metrics["total"]

        checkpoint_path = CHECKPOINT_DIR / "best_cross_attention_hybrid.pth"

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_metrics["total"],
                "history": history,
                "model_name": "CrossAttentionSpectraFusionNet_HybridLoss",
            },
            checkpoint_path
        )


print("✅ Best Cross-Attention Hybrid model saved")

elapsed = time.time() - start_time

print("\nTraining Complete")
print(f"Best Val Loss: {best_val_loss:.6f}")
print(f"Time: {elapsed / 60:.2f} minutes")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["train_total"], label="Train Total", linewidth=2)
plt.plot(history["val_total"], label="Validation Total", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Hybrid Loss")
plt.title("Cross-Attention Hybrid Loss Training Curve")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history["train_l1"], label="Train L1", linewidth=2)
plt.plot(history["val_l1"], label="Validation L1", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("L1 Loss")
plt.title("L1 Component")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history["train_ssim"], label="Train SSIM Loss", linewidth=2)
plt.plot(history["val_ssim"], label="Validation SSIM Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("SSIM Loss")
plt.title("SSIM Component")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history["train_physics"], label="Train Physics Loss", linewidth=2)
plt.plot(history["val_physics"], label="Validation Physics Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Physics Loss")
plt.title("Physics Consistency Component")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()